# Generating Rhode Island patient sample points

This notebook documents the methodology used to generate the Rhode Island patient location dataset consumed by `RI_simulations.jl` and `RI_STPMDP_ORS.jl`. The canonical 5,000-point output for the paper is checked in as **`../sampled_points/RI_points.csv`** — this notebook reproduces the methodology that generated it.

**Method.** Patient locations are sampled across Rhode Island, weighted by 2019 population density (Meta/CIESIN high-resolution data via the Humanitarian Data Exchange — same raster used for the California notebook, just masked to a different polygon). Sampled points are then constrained to land via spatial join against Rhode Island's US Census Bureau TIGER/Line **census tract** polygons (state FIPS 44).

## Why census tracts instead of ZIP codes (as in the CA notebook)?

The canonical `RI_points.csv` carries TIGER/Line census-tract columns (`STATEFP`, `COUNTYFP`, `TRACTCE`, `GEOID`, etc.), so the original pipeline used census tracts for Rhode Island even though it used a ZIP shapefile for the Bay Area. This is the documented methodological choice and is preserved here.

## Data sources (download separately)

These files are NOT committed to the repo because they are large. Download them into `data/` before running:

- **Population density (.vrt + companion .tif/.vrt tiles):** https://data.humdata.org/dataset/united-states-high-resolution-population-density-maps-demographic-estimates (same file used by `filter_points_CA.ipynb`)
- **Rhode Island census tract shapefile:** TIGER/Line `tl_YYYY_44_tract.shp` from https://www2.census.gov/geo/tiger/TIGER2020/TRACT/ (or whichever vintage year you prefer; the original used the 2019 or 2020 vintage)

See `README.md` in this folder for full setup instructions.


## Setup and parameters

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import Point
import matplotlib.pyplot as plt

# --- Paths (relative to this notebook's location: repo/data_prep/) ---
DATA_DIR = Path("data")
VRT_FILE = DATA_DIR / "population_usa_2019-07-01.vrt"
# Adjust the filename below to whatever TIGER/Line vintage you downloaded.
# RI's state FIPS code is 44, so the filename pattern is tl_<YEAR>_44_tract.shp.
RI_TRACTS_SHP = DATA_DIR / "tl_2020_44_tract.shp"

OUT_DIR = Path("../sampled_points")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_POINTS_CSV = OUT_DIR / "RI_points_raw.csv"
FILTERED_POINTS_CSV = OUT_DIR / "RI_points_regenerated.csv"  # does NOT overwrite canonical RI_points.csv

# --- Sampling parameters ---
NUM_SAMPLES = 5000
RANDOM_SEED = 42
DROP_DUPLICATES = True

# Seed both Python's `random` and NumPy's RNG.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Sampling {NUM_SAMPLES} points (seed={RANDOM_SEED}, drop_duplicates={DROP_DUPLICATES})")
print(f"Input population raster: {VRT_FILE}")
print(f"Input RI tracts:         {RI_TRACTS_SHP}")
print(f"Output (raw):            {RAW_POINTS_CSV}")
print(f"Output (filtered):       {FILTERED_POINTS_CSV}")

## Step 1 — Load population density data, masked to Rhode Island

In [ ]:
def get_population_data(vrt_file_path, shape_file):
    region_shape = gpd.read_file(shape_file)
    with rasterio.open(vrt_file_path) as src:
        if str(region_shape.crs) != src.crs.to_string():
            region_shape = region_shape.to_crs(src.crs)
        out_image, out_transform = mask(src, region_shape.geometry, crop=True)
        out_meta = src.meta.copy()
    population_data = out_image[0]
    no_data = out_meta["nodata"]
    population_data = np.where(population_data == no_data, np.nan, population_data)
    return population_data, out_transform

population_data, out_transform = get_population_data(VRT_FILE, RI_TRACTS_SHP)
print(f"Population raster shape: {population_data.shape}")
print(f"Non-NaN cells: {np.sum(~np.isnan(population_data)):,}")

## Step 2 — Weighted random sample of points

In [ ]:
def sample_points(population_data, out_transform, num_samples, rng):
    flat_population = population_data.flatten()
    indices = np.arange(flat_population.size)
    valid_indices = indices[flat_population > 0]
    weights = flat_population[valid_indices]
    weights /= weights.sum()
    sampled_indices = rng.choice(valid_indices, size=num_samples, p=weights)
    sampled_rows, sampled_cols = np.unravel_index(sampled_indices, population_data.shape)
    return [rasterio.transform.xy(out_transform, r, c) for r, c in zip(sampled_rows, sampled_cols)]

rng = np.random.default_rng(RANDOM_SEED)
sampled_points = sample_points(population_data, out_transform, NUM_SAMPLES, rng)
print(f"Sampled {len(sampled_points)} points")
print("First 3:", sampled_points[:3])

## Step 3 — Inspect duplicates, optionally drop them

In [ ]:
points_df = pd.DataFrame(sampled_points, columns=["longitude", "latitude"])

dup_mask = points_df.duplicated(subset=["longitude", "latitude"], keep=False)
print(f"Duplicate rows (including all repeats): {dup_mask.sum()}")

if DROP_DUPLICATES:
    before = len(points_df)
    points_df = points_df.drop_duplicates(subset=["longitude", "latitude"]).reset_index(drop=True)
    print(f"Dropped {before - len(points_df)} duplicate rows; {len(points_df)} unique points remain.")

points_df.to_csv(RAW_POINTS_CSV, index=False)
print(f"Wrote raw points (pre-land-filter) to {RAW_POINTS_CSV}")

## Step 4 — Filter to land via spatial join with census tracts

Spatial join against the RI census tract polygons. The `inner` join keeps the per-tract columns (STATEFP, COUNTYFP, TRACTCE, GEOID, NAME, ...), which matches the schema of the canonical `RI_points.csv`.

In [ ]:
geometry = [Point(xy) for xy in zip(points_df.longitude, points_df.latitude)]
points_gdf = gpd.GeoDataFrame(points_df.copy(), geometry=geometry)

tracts_gdf = gpd.read_file(RI_TRACTS_SHP)
if points_gdf.crs is None:
    points_gdf.set_crs(tracts_gdf.crs, inplace=True)
elif points_gdf.crs != tracts_gdf.crs:
    points_gdf = points_gdf.to_crs(tracts_gdf.crs)

# `op=` was deprecated in geopandas 0.10 and removed in 0.14; use `predicate=`.
points_on_land = gpd.sjoin(points_gdf, tracts_gdf, how="inner", predicate="intersects")
points_on_land = points_on_land.drop(columns="index_right")

print(f"Before land filter: {len(points_gdf)}")
print(f"After land filter:  {len(points_on_land)}")

# Match the column casing of the canonical RI_points.csv: Longitude/Latitude.
points_on_land = points_on_land.rename(columns={"longitude": "Longitude", "latitude": "Latitude"})

points_on_land.to_csv(FILTERED_POINTS_CSV, index=False)
print(f"Wrote land-filtered points to {FILTERED_POINTS_CSV}")
print("NOTE: This file is named `RI_points_regenerated.csv` so it does NOT overwrite the canonical `RI_points.csv`.")

## Step 5 — Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
tracts_gdf.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.3)
points_on_land.plot(ax=ax, marker="o", color="red", markersize=3, alpha=0.5)
ax.set_title(f"Rhode Island patient sample points (n={len(points_on_land)})")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()